# Supplementary Figure 12.4: Latent interpolation.

In [1]:
#| label: sfig12d_data

%matplotlib widget

import os
import random
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from tifffile import imread as tiff_imread
from urllib.parse import urlparse
import fsspec
from pathlib import Path

# Quiet logs
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TQDM_DISABLE"] = "1"
for _name in ("fsspec", "huggingface_hub", "urllib3", "datasets"):
    logging.getLogger(_name).setLevel(logging.ERROR)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, AveragePooling2D,
    Flatten, Dense, Reshape, ReLU
)

# ── Config ──────────────────────────────────────────────────────────────
REPO_ID    = "RussellBarkley/msa-em-figures"
BRANCH     = "main"
WEIGHT_DIR = Path("..") / "data" / "nucleus-ae"
ENC_NAME   = "encoder_weights.h5"
DEC_NAME   = "decoder_weights.h5"
LATENT_DIM = 512

N_PAIRS   = 50
T_VALUES  = np.linspace(0.0, 1.0, 6)

USE_SIMPLECACHE = True
RANDOM_SEED = 42

# ── Build models ────────────────────────────────────────────────────────
def build_encoder(latent_dim=LATENT_DIM):
    inp = Input((256, 256, 1), name="encoder_input")
    x = inp
    for filters in [16, 32, 64, 128]:
        x = Conv2D(filters, 3, padding="same")(x)
        x = ReLU()(x)
        x = Conv2D(filters, 3, padding="same")(x)
        x = ReLU()(x)
        x = AveragePooling2D(pool_size=2)(x)
    flat = Flatten()(x)
    z = Dense(latent_dim, name="z")(flat)
    return Model(inp, z, name="encoder")

def build_decoder(latent_dim=LATENT_DIM):
    z_in = Input((latent_dim,), name="z_sampling")
    x = Dense(16 * 16 * 128)(z_in)
    x = Reshape((16, 16, 128))(x)
    for filters in [128, 64, 32, 16]:
        x = Conv2DTranspose(filters, 3, strides=2, padding="same")(x)
        x = ReLU()(x)
        x = Conv2D(filters, 3, padding="same")(x)
        x = ReLU()(x)
    out = Conv2D(1, 3, padding="same", activation="sigmoid", name="decoder_output")(x)
    return Model(z_in, out, name="decoder")

# GPU memory growth
try:
    for g in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(g, True)
except Exception:
    pass

# ── Load models ─────────────────────────────────────────────────────────
encoder = build_encoder()
decoder = build_decoder()

enc_path = WEIGHT_DIR / ENC_NAME
dec_path = WEIGHT_DIR / DEC_NAME

if enc_path.exists() and dec_path.exists():
    encoder.load_weights(str(enc_path))
    decoder.load_weights(str(dec_path))

# ── HF dataset discovery ────────────────────────────────────────────────
fs_hf = fsspec.filesystem("hf")

def _ensure_hf_uri(p: str) -> str:
    return p if p.startswith("hf://") else ("hf://" + p.lstrip("/"))

def _maybe_cache(uri: str) -> str:
    return f"simplecache::{uri}" if USE_SIMPLECACHE else uri

def _list_all_tiff_paths():
    globs = [
        f"hf://datasets/{REPO_ID}@{BRANCH}/**/*.tif",
        f"hf://datasets/{REPO_ID}@{BRANCH}/**/*.tiff",
    ]
    paths = []
    for pat in globs:
        try:
            for p in fs_hf.glob(pat):
                paths.append(_ensure_hf_uri(p))
        except Exception:
            pass
    return sorted(set(paths))

ALL_PATHS = _list_all_tiff_paths()
MODE = "files" if len(ALL_PATHS) > 0 else "dataset"

DS_OBJ = None
DS_LEN = 0
if MODE == "dataset":
    from datasets import load_dataset
    DS_OBJ = load_dataset(REPO_ID, split="train", keep_in_memory=False)
    DS_LEN = len(DS_OBJ)

# ── Image loading helpers ───────────────────────────────────────────────
def normalize01(x):
    x = x.astype(np.float32)
    return x / 255.0 if x.max() > 1.0 else x

def to_grayscale(img):
    """Convert image to grayscale if needed."""
    if img.ndim == 3:
        if img.shape[-1] == 1:
            return img[..., 0]
        elif img.shape[-1] == 3:
            # RGB to grayscale using standard weights
            return 0.2126 * img[..., 0] + 0.7152 * img[..., 1] + 0.0722 * img[..., 2]
        else:
            return img[..., 0]
    return img

def load_img_file_hf(uri: str):
    with fsspec.open(_maybe_cache(uri), "rb") as f:
        img = tiff_imread(f)
    img = to_grayscale(img)
    if img.shape != (256, 256):
        raise ValueError(f"Image has shape {img.shape}, expected 256x256.")
    return normalize01(img)

def load_img_dataset_idx(idx: int):
    ex = DS_OBJ[int(idx)]
    arr = np.asarray(ex["image"])
    arr = to_grayscale(arr)
    if arr.shape != (256, 256):
        raise ValueError(f"Image has shape {arr.shape}, expected 256x256.")
    return normalize01(arr)

def load_handle(h):
    """Load image from either file URI or dataset index."""
    if MODE == "files":
        return load_img_file_hf(h)
    else:
        return load_img_dataset_idx(int(h))

# ── Encoding/decoding ───────────────────────────────────────────────────
def encode_img(img01):
    return encoder.predict(img01[None, ..., None], batch_size=8, verbose=0)[0]

def decode_many(z_batch):
    rec = decoder.predict(z_batch, batch_size=min(len(z_batch), 16), verbose=0)[..., 0]
    return np.clip(rec, 0.0, 1.0)

# ── Pair generation ─────────────────────────────────────────────────────
rng_py = random.Random(RANDOM_SEED)

def make_pairs(n_pairs=N_PAIRS):
    """Generate random pairs from dataset."""
    N = len(ALL_PATHS) if MODE == "files" else DS_LEN
    if N < 2:
        return []
    
    # Sample indices
    m = min(2 * n_pairs, N)
    indices = list(range(N))
    rng_py.shuffle(indices)
    chosen = indices[:m]
    
    # Create pairs
    pairs = []
    for i in range(0, m - (m % 2), 2):
        if MODE == "files":
            pairs.append((ALL_PATHS[chosen[i]], ALL_PATHS[chosen[i+1]]))
        else:
            pairs.append((chosen[i], chosen[i+1]))
    
    # Fill remaining pairs if needed
    while len(pairs) < n_pairs:
        a = rng_py.randrange(N)
        b = rng_py.randrange(N)
        if MODE == "files":
            pairs.append((ALL_PATHS[a], ALL_PATHS[b]))
        else:
            pairs.append((a, b))
    
    return pairs

# ── Figure setup ────────────────────────────────────────────────────────
_was_interactive = plt.isinteractive()
plt.ioff()

K = len(T_VALUES)
fig = plt.figure(figsize=(4.4, 2.0), dpi=150, constrained_layout=False)
gs = fig.add_gridspec(1, K)

axes, ims = [], []
for k in range(K):
    ax = fig.add_subplot(gs[0, k])
    im = ax.imshow(np.zeros((256, 256)), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)
    ax.set_title(f"t={T_VALUES[k]:.2f}", fontsize=6, pad=2)
    axes.append(ax)
    ims.append(im)

fig.subplots_adjust(left=0.01, right=0.99, bottom=0.02, top=0.88, wspace=0.02)
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.layout.width = "auto"
fig.canvas.layout.height = "auto"

if _was_interactive:
    plt.ion()

# ── State & cache ───────────────────────────────────────────────────────
pairs = []
pair_cache = {}  # idx -> frames array

def show_frames(frames):
    for k, im in enumerate(ims):
        im.set_data(frames[k])
    fig.canvas.draw_idle()

def compute_pair(idx):
    if idx in pair_cache:
        return pair_cache[idx]
    
    a_h, b_h = pairs[idx]
    A = load_handle(a_h)
    B = load_handle(b_h)
    A_z = encode_img(A)
    B_z = encode_img(B)
    
    # Interpolate in latent space
    Z = np.stack([(1.0 - t) * A_z + t * B_z for t in T_VALUES], axis=0)
    frames = decode_many(Z)
    
    pair_cache[idx] = frames
    return frames

# ── Widgets ─────────────────────────────────────────────────────────────
w_rescan = W.Button(description="Resample pairs", button_style="info")
w_pair = W.IntSlider(
    value=0, min=0, max=N_PAIRS-1, step=1,
    description="Pair #", readout=True, continuous_update=False
)
w_prev = W.Button(description="Prev")
w_next = W.Button(description="Next")

def on_rescan_clicked(_):
    global pairs, pair_cache
    pairs = make_pairs(N_PAIRS)
    pair_cache = {}
    
    if pairs:
        w_pair.max = len(pairs) - 1
        w_pair.value = 0
        frames = compute_pair(0)
        show_frames(frames)

def on_pair_change(change):
    idx = int(change["new"])
    if pairs:
        frames = compute_pair(idx)
        show_frames(frames)

def on_prev(_):
    if pairs:
        w_pair.value = max(0, w_pair.value - 1)

def on_next(_):
    if pairs:
        w_pair.value = min(w_pair.max, w_pair.value + 1)

w_rescan.on_click(on_rescan_clicked)
w_pair.observe(on_pair_change, names="value")
w_prev.on_click(on_prev)
w_next.on_click(on_next)

# ── Display ─────────────────────────────────────────────────────────────
container = W.VBox([
    W.HBox([w_rescan, w_prev, w_next, w_pair]),
    fig.canvas
])
display(container)

# Initial load
on_rescan_clicked(None)